# 03 IndoBERT Sentiment Modeling


## CRISP-DM Stage
Modeling.

## Research Context
Phase 8B adds a controlled IndoBERT fine-tuning entrypoint after Phase 8A prepared train, validation, and test splits. This notebook documents the training setup, dry-run validation, expected metrics, and local artifact policy. Full fine-tuning is not executed automatically.

## IndoBERT Role in SentiRank
IndoBERT is used only for sentiment analysis. The model predicts `Negative`, `Neutral`, and `Positive` labels from preprocessed Indonesian review text. Aspect classification remains a separate SVM workflow and is not trained in this phase.

## Input Dataset
Primary input:
- `datasets/processed/reviews_final.csv`

Expected columns include `external_id`, `rating`, `content`, `initial_sentiment`, `final_sentiment`, `text_indobert`, `text_svm`, `reviewed_at`, `source`, and `app_id`.

## Target Label
The target label is `final_sentiment`, produced by the conservative relabeling workflow. Allowed labels are `Negative`, `Neutral`, and `Positive`.

## Label Mapping
| Sentiment label | Label ID |
| --- | ---: |
| Negative | 0 |
| Neutral | 1 |
| Positive | 2 |

## Dataset Validation
The preparation script validates required columns, keeps only allowed labels, removes empty `text_indobert` rows, adds `label_id`, and preserves useful source columns for traceability.

## Stratified Split Strategy
The dataset is split using stratified sampling to preserve sentiment label proportions across train, validation, and test sets. Default split fractions are 70% train, 15% validation, and 15% test with `random_state=42`.

## Controlled Fine-Tuning Script
The training entrypoint is `ml-service/scripts/train_indobert.py`. It supports a `--dry-run` mode that validates split files, required columns, label mapping, dataset sizes, label distributions, and planned configuration without importing model classes, downloading weights, training, or writing artifacts.

## Google Colab Training Workflow
Actual IndoBERT fine-tuning should be executed with GPU through `ml-service/notebooks/03a_indobert_colab_training.ipynb`. The local `train_indobert.py` script remains useful for reproducibility and dry-run validation, but local full training is intentionally avoided.

## IndoBERT Experiment Runs
Run 1 baseline has been completed in Google Colab. Run 2 weighted-loss has also been completed and improved Neutral recall and macro F1. Run 3 lower-learning-rate weighted-loss is prepared in `ml-service/notebooks/03c_indobert_weighted_loss_lr1e5_colab_training.ipynb` to test whether `learning_rate = 1e-5` improves the trade-off.

## Dependency Policy
The controlled training path uses PyTorch, Hugging Face Transformers, Datasets, Evaluate, Accelerate, scikit-learn, pandas, and matplotlib. Heavy ML imports are lazy-loaded only for non-dry-run training so validation can run without model initialization.

## Training Configuration Plan
Default controlled fine-tuning configuration:
- `model_name`: `indobenchmark/indobert-base-p1`
- `max_length`: 128
- `batch_size`: 16
- `learning_rate`: 2e-5
- `epochs`: 3
- `evaluation_strategy`: epoch
- `save_strategy`: epoch
- `metric_for_best_model`: f1_macro
- `random_state`: 42

## Evaluation Plan
The training script computes accuracy, macro precision, macro recall, macro F1, weighted precision, weighted recall, and weighted F1 on the test split. Macro F1 is the primary model-selection metric because the relabeled dataset is not perfectly balanced.

## Expected Model Artifact Output
When full training is explicitly approved and executed, the fine-tuned model and tokenizer are saved under `ml-service/saved_models/indobert/`. These files are local generated artifacts and must not be committed.

## Expected Dataset Outputs
Generated local split files:
- `datasets/processed/indobert/train.csv`
- `datasets/processed/indobert/validation.csv`
- `datasets/processed/indobert/test.csv`
- `datasets/processed/indobert/label_mapping.json`

These files are generated local artifacts and remain ignored by Git.

## Expected Metrics Output
Preparation metrics from Phase 8A:
- `datasets/outputs/eda/03_indobert/indobert_dataset_summary.json`
- `datasets/outputs/eda/03_indobert/indobert_label_distribution.csv`
- `datasets/outputs/eda/03_indobert/indobert_label_distribution.json`
- `datasets/outputs/eda/03_indobert/indobert_split_distribution.csv`
- `datasets/outputs/eda/03_indobert/indobert_split_distribution.json`
- `datasets/outputs/eda/03_indobert/indobert_text_length_summary.json`

Training metrics expected after approved full fine-tuning:
- `datasets/outputs/eda/03_indobert/indobert_training_metrics.json`
- `datasets/outputs/eda/03_indobert/indobert_test_predictions.csv`

## Figure Outputs
Preparation figures from Phase 8A:
- `docs/figures/03_indobert/indobert_label_distribution.png`
- `docs/figures/03_indobert/indobert_split_distribution.png`
- `docs/figures/03_indobert/indobert_text_length_distribution.png`

Training figures expected after approved full fine-tuning:
- `docs/figures/03_indobert/indobert_training_loss.png`
- `docs/figures/03_indobert/indobert_eval_metrics.png`

## Reproducible Command
Run dry-run validation from `ml-service/`:

```bash
uv run python scripts/train_indobert.py --dry-run
```

Full training uses the same script without `--dry-run`, but it should only be executed after explicit approval because it downloads model weights and runs a long training job.

## Local Artifact Policy
Generated split CSV files, prediction CSV files, and model artifacts are local outputs. The trained model directory under `ml-service/saved_models/indobert/` and processed split files under `datasets/processed/indobert/` remain ignored by Git.

## Expected Full Training Command
Run from `ml-service/` only after approval:

```bash
uv run python scripts/train_indobert.py
```

## Limitations
Phase 8B implements the training path and validates it in dry-run mode only. The current labels are still derived from rating and conservative keyword relabeling, so later predictive performance must be interpreted as model performance against weakly corrected sentiment labels, not manual gold-standard annotation.

## Next Step
After explicit approval, run full IndoBERT fine-tuning and then continue to `05_model_evaluation.ipynb` for model evaluation synthesis.